In [1]:
from botasaurus.browser import Driver
from botasaurus.soupify import soupify
from bs4 import BeautifulSoup
from urllib.parse import urlparse, parse_qs
from pathlib import Path
from dotenv import load_dotenv
from random import randint
import pandas as pd 
import re, os, time, sys, csv, json
from datetime import datetime, timedelta
from toolkit import ordergenerator as og
from toolkit import general_tools as gt

In [2]:
driver = Driver()

In [3]:
"https://www.booking.com/hotel/es/parador-de-turismo-de-antequera.fr.html?aid=311089&label=parador-de-turismo-de-antequera-mb%2AsFegnwfvsw8iVRqa5rQS461046900414%3Apl%3Ata%3Ap1%3Ap2%3Aac%3Aap%3Aneg%3Afi%3Atikwd-21638814841%3Alp9054903%3Ali%3Adec%3Adm&sid=3e75a03babc1c19066249b8dd288ecce&age=1&checkin=2025-04-07&checkout=2025-04-11&dest_id=-370891&dest_type=city&dist=0&group_adults=2&group_children=1&hapos=1&hpos=1&no_rooms=1&req_adults=2&req_age=1&req_children=1&room1=A%2CA%2C1&sb_price_type=total&soh=1&sr_order=popularity&srepoch=1743688831&srpvid=1a21627e834c0198&type=total&ucfs=1&#no_availability_msg"

"https://www.booking.com/searchresults.fr.html?aid=311089&label=hotel-es-96997-fr-O9PxBSUXmnjLs1Y41XHXuAS261055391465%3Apl%3Ata%3Ap1%3Ap2%3Aac%3Aap%3Aneg%3Afi%3Atikwd-354819127664%3Alp9054903%3Ali%3Adec%3Adm%3Appccp%3DUmFuZG9tSVYkc2RlIyh9Yf23yREhrOV9DIzdQnby_R8&gclid=CjwKCAjw47i_BhBTEiwAaJfPpgCoSOmqD7-ToZD7aYM4vrXyogKrxS-7fUq7KTdtn0l_2FcRzmpg3BoC4YcQAvD_BwE&highlighted_hotels=96997&checkin=2025-04-07&redirected=1&city=-370891&hlrd=with_av&source=hotel&checkout=2025-04-11&keep_landing=1&sid=3e75a03babc1c19066249b8dd288ecce"

"https://www.booking.com/hotel/es/la-sierra.fr.html?aid=311089&label=hotel-97366-es-WYimCqUgSsIz3ehNUtd3swS384251002520%3Apl%3Ata%3Ap1%3Ap2%3Aac%3Aap%3Aneg%3Afi%3Atikwd-357129280767%3Alp9054903%3Ali%3Adec%3Adm%3Appccp%3DUmFuZG9tSVYkc2RlIyh9Yf23yREhrOV9DIzdQnby_R8&sid=3e75a03babc1c19066249b8dd288ecce&age=1&all_sr_blocks=9736609_333934460_0_34_0&checkin=2025-04-07&checkout=2025-04-11&dest_id=-370891&dest_type=city&dist=0&group_adults=2&group_children=1&hapos=1&highlighted_blocks=9736609_333934460_0_34_0&hpos=1&matching_block_id=9736609_333934460_0_34_0&no_rooms=1&req_adults=2&req_age=1&req_children=1&room1=A%2CA%2C1&sb_price_type=total&sr_order=popularity&sr_pri_blocks=9736609_333934460_0_34_0__32000&srepoch=1743688892&srpvid=6c45629cfb67092c&type=total&ucfs=1&"

"https://www.booking.com/hotel/es/lozano.fr.html?aid=356982&label=gog235jc-1DCAsoRkIGbG96YW5vSA1YA2hNiAEBmAENuAEXyAEP2AED6AEB-AECiAIBqAIDuAKKqLq_BsACAdICJDY0YTRlN2NjLTE1YTAtNGM5ZC1iZjkwLWY1YzY5Njc0OWI0NtgCBOACAQ&sid=3e75a03babc1c19066249b8dd288ecce&all_sr_blocks=2586806_287749578_2_2_0&checkin=2025-04-07&checkout=2025-04-11&dest_id=-370891&dest_type=city&dist=0&group_adults=2&group_children=0&hapos=1&highlighted_blocks=2586806_287749578_2_2_0&hpos=1&matching_block_id=2586806_287749578_2_2_0&no_rooms=1&req_adults=2&req_children=0&room1=A%2CA&sb_price_type=total&sr_order=popularity&sr_pri_blocks=2586806_287749578_2_2_0__33200&srepoch=1743688743&srpvid=13ca6250db8d074c&type=total&ucfs=1&"

"https://www.booking.com/hotel/es/los-dolmenes.fr.html?label=New_English_EN_FR_21457884505-T0t*edlXm_oGkIMqMNUhEAS217291026622%3Apl%3Ata%3Ap1%3Ap2%3Aac%3Aap%3Aneg&sid=3e75a03babc1c19066249b8dd288ecce&gclid=CjwKCAjw47i_BhBTEiwAaJfPpppWw-B7lKrfTsDI_1uDHyySten0DS2Te1ybHrDj9urrZPt3fTuIwxoC5BUQAvD_BwE&aid=318615&ucfs=1&arphpl=1&checkin=2025-04-07&checkout=2025-04-11&dest_id=-370891&dest_type=city&group_adults=2&req_adults=2&no_rooms=1&group_children=1&req_children=1&age=1&req_age=1&hpos=1&hapos=1&sr_order=popularity&srpvid=721962a7618d03db&srepoch=1743688914&all_sr_blocks=9718802_408831757_2_2_0&highlighted_blocks=9718802_408831757_2_2_0&matching_block_id=9718802_408831757_2_2_0&sr_pri_blocks=9718802_408831757_2_2_0__36720&from=searchresults"

"https://www.booking.com/hotel/es/abades-el-mirador-loja.fr.html?aid=318615&label=New_English_EN_FR_21457884505-T0t%2AedlXm_oGkIMqMNUhEAS217291026622%3Apl%3Ata%3Ap1%3Ap2%3Aac%3Aap%3Aneg&sid=3e75a03babc1c19066249b8dd288ecce&age=1&checkin=2025-04-07&checkout=2025-04-11&dest_id=-389604&dest_type=city&dist=0&group_adults=2&group_children=1&hapos=1&hpos=1&no_rooms=1&req_adults=2&req_age=1&req_children=1&room1=A%2CA%2C1&sb_price_type=total&soh=1&sr_order=popularity&srepoch=1743688944&srpvid=920162b64bc400a5&type=total&ucfs=1&#no_availability_msg"

'https://www.booking.com/hotel/es/abades-el-mirador-loja.fr.html?aid=318615&label=New_English_EN_FR_21457884505-T0t%2AedlXm_oGkIMqMNUhEAS217291026622%3Apl%3Ata%3Ap1%3Ap2%3Aac%3Aap%3Aneg&sid=3e75a03babc1c19066249b8dd288ecce&age=1&checkin=2025-04-07&checkout=2025-04-11&dest_id=-389604&dest_type=city&dist=0&group_adults=2&group_children=1&hapos=1&hpos=1&no_rooms=1&req_adults=2&req_age=1&req_children=1&room1=A%2CA%2C1&sb_price_type=total&soh=1&sr_order=popularity&srepoch=1743688944&srpvid=920162b64bc400a5&type=total&ucfs=1&#no_availability_msg'

In [4]:
url = "https://www.booking.com/hotel/es/apartamento-calle-purgatorio-centro.fr.html?aid=311089&label=hotel-es-96997-fr-O9PxBSUXmnjLs1Y41XHXuAS261055391465%3Apl%3Ata%3Ap1%3Ap2%3Aac%3Aap%3Aneg%3Afi%3Atikwd-354819127664%3Alp9054903%3Ali%3Adec%3Adm%3Appccp%3DUmFuZG9tSVYkc2RlIyh9Yf23yREhrOV9DIzdQnby_R8&sid=a74f46564b53eaf0816f2bb1bee55c3a&all_sr_blocks=573664802_381603430_2_0_0&checkin=2025-04-10&checkout=2025-04-11&dest_id=-370891&dest_type=city&dist=0&group_adults=2&group_children=0&hapos=24&highlighted_blocks=573664802_381603430_2_0_0&hpos=24&matching_block_id=573664802_381603430_2_0_0&no_rooms=1&req_adults=2&req_children=0&room1=A%2CA&sb_price_type=total&sr_order=popularity&sr_pri_blocks=573664802_381603430_2_0_0__18221&srepoch=1744224866&srpvid=501f6fd41e32083f&type=total&ucfs=1"

In [5]:
driver.get(url)

<Tab [AFBF6FF41330096E7321C2289F9E8F15] [page] [url: https://www.booking.com/hotel/es/apartamento-calle-purgatorio-centro.fr.html?aid=311089&label=hotel-es-96997-fr-O9PxBSUXmnjLs1Y41XHXuAS261055391465%3Apl%3Ata%3Ap1%3Ap2%3Aac%3Aap%3Aneg%3Afi%3Atikwd-354819127664%3Alp9054903%3Ali%3Adec%3Adm%3Appccp%3DUmFuZG9tSVYkc2RlIyh9Yf23yREhrOV9DIzdQnby_R8&sid=a74f46564b53eaf0816f2bb1bee55c3a&all_sr_blocks=573664802_381603430_2_0_0&checkin=2025-04-10&checkout=2025-04-11&dest_id=-370891&dest_type=city&dist=0&group_adults=2&group_children=0&hapos=24&highlighted_blocks=573664802_381603430_2_0_0&hpos=24&matching_block_id=573664802_381603430_2_0_0&no_rooms=1&req_adults=2&req_children=0&room1=A%2CA&sb_price_type=total&sr_order=popularity&sr_pri_blocks=573664802_381603430_2_0_0__18221&srepoch=1744224866&srpvid=501f6fd41e32083f&type=total&ucfs=1]>

In [32]:
page = BeautifulSoup(driver.page_html.encode('utf-8').decode('utf-8'), 'html.parser')

In [33]:
nom = page.find('h2', {'class':'d2fee87262 pp-header__title'}).text
print(f'nom: {nom}')

nom: Apartamento calle Purgatorio Centro


In [8]:
localite = page.find('div', {'class':'a53cbfa6de f17adf7576'})
localite_other_content = localite.find('div', {'class':'ac52cd96ed'}).text
localite = localite.text.split(localite_other_content)[0].strip().replace(',', ' -')

In [ ]:
container = page.find('table', {'id':'hprt-table'})
bool(container)


True

In [10]:
t_body = container.find('tbody')

In [11]:
rows = t_body.find_all('tr')
len(rows)

2

In [12]:
typologie = rows[0].find('th').find('span', {'class':'hprt-roomtype-icon-link'}).text.replace('\n', '').split('(')[0].strip()
typologie

'Appartement Supérieur 2 Chambres'

In [13]:
prix_actuel = int(rows[0]["data-hotel-rounded-price"])
prix_actuel

182

In [36]:
rows[0].find('div', {'class':'bui-f-color-destructive js-strikethrough-price prco-inline-block-maker-helper bui-price-display__original'})['data-strikethrough-value']

'202.46'

In [14]:
taxe = rows[0].find('td', {'class':'hp-price-left-align hprt-table-cell hprt-table-cell-price'})
taxe_text = taxe.find('div', class_='prd-taxes-and-fees-under-price').text
taxe_value = 0
try:
    taxe_value =  int(''.join(filter(str.isdigit, taxe_text)))
except:
    pass    

In [15]:
prix_actuel += taxe_value

In [16]:
cleaned_data = []

typologie = ""


for row in rows:
    data = {}
    data["nom"] = nom 
    if row.find('th'):
        typologie = row.find('th').find('span', {'class':'hprt-roomtype-icon-link'}).text.replace('\n', '').split('(')[0].strip()
    taxe = rows[0].find('td', {'class':'hp-price-left-align hprt-table-cell hprt-table-cell-price'})
    taxe_text = taxe.find('div', class_='prd-taxes-and-fees-under-price').text
    taxe_value = 0
    try:
        taxe_value =  int(''.join(filter(str.isdigit, taxe_text)))
    except:
        pass    
    taxe_text = 0 if 'compri' not in taxe_text else taxe_value
    data["prix_actuel"] = row["data-hotel-rounded-price"]
    data["prix_init"] = int(data['prix_actuel']) + taxe_value
    data["localite"] = localite
    data["typologie"] = typologie
    cleaned_data.append(data)

    

In [17]:
df = pd.DataFrame(cleaned_data)
df.head()

,nom,prix_actuel,prix_init,localite,typologie
0,Apartamento calle Purgatorio Centro,182,219,calle purgatorio - 2 - bajo K - 29200 Antequer...,Appartement Supérieur 2 Chambres
1,Apartamento calle Purgatorio Centro,213,250,calle purgatorio - 2 - bajo K - 29200 Antequer...,Appartement Supérieur 2 Chambres


In [18]:
len(df)

2

In [19]:
len(df.drop_duplicates())

2

In [20]:
new_df = df.drop_duplicates()

In [21]:
new_df.to_csv("simple_data.csv", index=False)

In [22]:
url1 = "https://www.booking.com/hotel/es/parador-de-turismo-de-antequera.fr.html?aid=311089&label=parador-de-turismo-de-antequera-mb%2AsFegnwfvsw8iVRqa5rQS461046900414%3Apl%3Ata%3Ap1%3Ap2%3Aac%3Aap%3Aneg%3Afi%3Atikwd-21638814841%3Alp9054903%3Ali%3Adec%3Adm&sid=3e75a03babc1c19066249b8dd288ecce&age=1&checkin=2025-04-09&checkout=2025-04-11&dest_id=-370891&dest_type=city&dist=0&group_adults=2&group_children=1&hapos=1&hpos=1&no_rooms=1&req_adults=2&req_age=1&req_children=1&room1=A%2CA%2C1&sb_price_type=total&soh=1&sr_order=popularity&srepoch=1743688831&srpvid=1a21627e834c0198&type=total&ucfs=1&#no_availability_msg"


url2 = "https://www.booking.com/hotel/es/parador-de-turismo-de-antequera.fr.html?aid=311089&label=parador-de-turismo-de-antequera-mb%2AsFegnwfvsw8iVRqa5rQS461046900414%3Apl%3Ata%3Ap1%3Ap2%3Aac%3Aap%3Aneg%3Afi%3Atikwd-21638814841%3Alp9054903%3Ali%3Adec%3Adm&sid=3e75a03babc1c19066249b8dd288ecce&dest_id=-370891&dest_type=city&dist=0&group_adults=2&group_children=1&hapos=1&hpos=1&no_rooms=1&req_adults=2&req_age=1&req_children=1&room1=A%2CA%2C1&sb_price_type=total&soh=1&sr_order=popularity&srepoch=1743688831&srpvid=1a21627e834c0198&type=total&ucfs=1&#no_availability_msg&checkin=2025-04-09&checkout=2025-04-10&selected_currency=EUR&lang=fr"

In [23]:
query_order = [
    "aid",
    "label",
    "sid",
    "age",
    "checkin",
    "checkout",
    "dest_id",
    "dest_type",
    "dist",
    "group_children",
    "hapos",
    "hpos",
    "no_rooms",
    "req_adults",
    "req_age",
    "req_children",
    "room1",
    "sb_price_type",
    "soh",
    "sr_order",
    "srepoch",
    "srpvid",
    "type",
    "ucfs"
]

In [24]:
parse_qs(url.split('?')[-1])

{'aid': ['311089'],
 'label': ['hotel-es-96997-fr-O9PxBSUXmnjLs1Y41XHXuAS261055391465:pl:ta:p1:p2:ac:ap:neg:fi:tikwd-354819127664:lp9054903:li:dec:dm:ppccp=UmFuZG9tSVYkc2RlIyh9Yf23yREhrOV9DIzdQnby_R8'],
 'sid': ['a74f46564b53eaf0816f2bb1bee55c3a'],
 'all_sr_blocks': ['573664802_381603430_2_0_0'],
 'checkin': ['2025-04-10'],
 'checkout': ['2025-04-11'],
 'dest_id': ['-370891'],
 'dest_type': ['city'],
 'dist': ['0'],
 'group_adults': ['2'],
 'group_children': ['0'],
 'hapos': ['24'],
 'highlighted_blocks': ['573664802_381603430_2_0_0'],
 'hpos': ['24'],
 'matching_block_id': ['573664802_381603430_2_0_0'],
 'no_rooms': ['1'],
 'req_adults': ['2'],
 'req_children': ['0'],
 'room1': ['A,A'],
 'sb_price_type': ['total'],
 'sr_order': ['popularity'],
 'sr_pri_blocks': ['573664802_381603430_2_0_0__18221'],
 'srepoch': ['1744224866'],
 'srpvid': ['501f6fd41e32083f'],
 'type': ['total'],
 'ucfs': ['1']}

In [25]:
b_date = "09/04/2025"
e_date = "11/04/2025"
freq = 1

In [26]:
start_date = datetime.strptime(b_date, "%d/%m/%Y")
end_date = datetime.strptime(e_date, "%d/%m/%Y")
date_space = int((end_date - start_date).days) + 1
date_space

3

In [27]:
from datetime import datetime, timedelta

station_url_test = "https://www.booking.com/hotel/es/parador-de-turismo-de-antequera.fr.html?aid=311089&label=parador-de-turismo-de-antequera-mb%2AsFegnwfvsw8iVRqa5rQS461046900414%3Apl%3Ata%3Ap1%3Ap2%3Aac%3Aap%3Aneg%3Afi%3Atikwd-21638814841%3Alp9054903%3Ali%3Adec%3Adm&sid=3e75a03babc1c19066249b8dd288ecce&dest_id=-370891&dest_type=city&dist=0&group_adults=2&group_children=1&hapos=1&hpos=1&no_rooms=1&req_adults=2&req_age=1&req_children=1&room1=A%2CA%2C1&sb_price_type=total&soh=1&sr_order=popularity&srepoch=1743688831&srpvid=1a21627e834c0198&type=total&ucfs=1&#no_availability_msg"


def generate_url(stations_url:list, freq:int, start_date:str, end_date:str) -> list:

    def normalize_url_params(url:str, start:str, end:str) -> str:
        """ normalize url parameters as needed for data scraping format """
        print(url)
        url_params = parse_qs(urlparse(url).query)
        if "checkin" not in url_params:
            url += f"&checkin={start}"
        if "checkout" not in url_params:
            url += f"&checkout={end}"
        if "selected_currency" not in url_params:
            url += "&selected_currency=EUR"
        if "lang" not in url_params:
            url += f"&lang=fr"
        return url
    

    """generate dynamic urls for any station between interval of given dates {start_date and end_date}"""
    time.sleep(1)
    correct_dest_url = []
    if freq in [1, 3, 7]:
        date_space = int((end_date - start_date).days) + 1
        checkin = start_date
        checkout = checkin + timedelta(days=freq)  

        for _ in range(date_space):
            for station_url in stations_url:
                url = normalize_url_params(station_url, checkin.strftime("%Y-%m-%d"), checkout.strftime("%Y-%m-%d"))
                correct_dest_url.append(url)
            checkin += timedelta(days=1)
            checkout += timedelta(days=1)

        return correct_dest_url

In [28]:
def validate_urls(url:str) -> str:
    global query_order
    base_url = url.split('?')[0]
    params = url.split('?')[-1]
    formated_ordered_params = ""
    query_url = parse_qs(params)
    for query in query_order:
        if bool(query_url.get(query)):
            formated_ordered_params += f"{query}={query_url.get(query, '')[0]}&"
    parms_keys = list(query_url.keys())
    new_params = [i for i in parms_keys if i not in query_order]
    for query in new_params:
        formated_ordered_params += f"{query}={query_url.get(query, '')[0]}&"
    return f"{base_url}?{formated_ordered_params}"[:-1]
        


In [29]:
test_url = "https://www.booking.com/hotel/es/parador-de-turismo-de-antequera.fr.html?aid=311089&label=parador-de-turismo-de-antequera-mb%2AsFegnwfvsw8iVRqa5rQS461046900414%3Apl%3Ata%3Ap1%3Ap2%3Aac%3Aap%3Aneg%3Afi%3Atikwd-21638814841%3Alp9054903%3Ali%3Adec%3Adm&sid=3e75a03babc1c19066249b8dd288ecce&dest_id=-370891&dest_type=city&dist=0&group_adults=2&group_children=1&hapos=1&hpos=1&no_rooms=1&req_adults=2&req_age=1&req_children=1&room1=A%2CA%2C1&sb_price_type=total&soh=1&sr_order=popularity&srepoch=1743688831&srpvid=1a21627e834c0198&type=total&ucfs=1&#no_availability_msg&checkin=2025-04-09&checkout=2025-04-10&selected_currency=EUR&lang=fr"

validate_urls(test_url)

'https://www.booking.com/hotel/es/parador-de-turismo-de-antequera.fr.html?aid=311089&label=parador-de-turismo-de-antequera-mb*sFegnwfvsw8iVRqa5rQS461046900414:pl:ta:p1:p2:ac:ap:neg:fi:tikwd-21638814841:lp9054903:li:dec:dm&sid=3e75a03babc1c19066249b8dd288ecce&checkin=2025-04-09&checkout=2025-04-10&dest_id=-370891&dest_type=city&dist=0&group_children=1&hapos=1&hpos=1&no_rooms=1&req_adults=2&req_age=1&req_children=1&room1=A,A,1&sb_price_type=total&soh=1&sr_order=popularity&srepoch=1743688831&srpvid=1a21627e834c0198&type=total&ucfs=1&group_adults=2&selected_currency=EUR&lang=fr'

In [30]:
parms = parse_qs(test_url.split("?")[-1])
parms_keys = list(parms.keys())
parms_keys

[i for i in parms_keys if i not in query_order]

['group_adults', 'selected_currency', 'lang']